# ML Challenge 2026 — Full-data run (Exp5)

**Goal:** blocking → train → inference on FULL data → `matching_results.tsv` → leaderboard.

**Time budget (free Colab, mostly unattended):** dry run ~10 min, test blocking ~2–5 h,
train-subset blocking ~1–2 h, training ~1 h, inference ~2–3 h. Leave the tab open;
every long cell prints progress + ETA.

**If the runtime dies:** re-run from the top — `git lfs pull` and every `--resume`
blocking cell skip finished work. Files already copied to Drive survive.

**Final config (locked on sample, Exp5a/5b):** char (3,4), min_df=2, max_df=0.05,
K=40/channel (val F0.5 0.9784 @ tau retuned on full-train val).

In [ ]:
# 0) Setup: Drive, repo (clone once, pull after), data, deps.
from google.colab import drive
drive.mount('/content/drive')
!if [ -d ML-challenge ]; then cd ML-challenge && git checkout arena/01a0df98-ml-challenge && git pull; else git clone -b arena/01a0df98-ml-challenge https://github.com/Saadmadni84/ML-challenge.git && cd ML-challenge; fi
%cd /content/ML-challenge
!git lfs pull
!ls -lh dataset/train dataset/test
!pip -q install -r code/business_entity_resolution/requirements.txt
!python3 -c "import sklearn, lightgbm, rapidfuzz, sparse_dot_topn; print('deps OK')"

In [ ]:
# 1) DRY RUN: 3k test queries/country against FULL targets (~10 min).
# Measures: index build time, retrieval S1/s, est. full runtime. PASTE OUTPUT.
S = "code/business_entity_resolution/src"
!python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field name --out output/dry_name_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US,France --max-q 3000
!python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field address --out output/dry_addr_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US,France --max-q 3000

⏸️ **PAUSE — send the dry-run output (index sizes, S1/s, ETAs) to chat.**
We confirm the full-run time budget before launching the hours-long cells.
If told to proceed, continue below.

In [ ]:
# 2) FULL TEST BLOCKING, per country+field with --resume (hours; ETA printed).
# Run the whole cell and leave the tab open. Re-runnable: skips finished shards.
S = "code/business_entity_resolution/src"
for C in ["India", "US", "France"]:
    for F in ["name", "address"]:
        print(f"=== {C} / {F} ===", flush=True)
        !python3 $S/blocking.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --field $F --out output/test_{country}_{field}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries $C --resume
        !cp output/test_{country}_{field}.tsv "/content/drive/MyDrive/test_{country}_{field}.tsv"
!ls -lh output/test_*.tsv

In [ ]:
# 3) Test unions (fast) + backup to Drive.
S = "code/business_entity_resolution/src"
!for C in India US France; do python3 $S/union_candidates.py --a output/test_${C}_name.tsv --b output/test_${C}_addr.tsv --k-per-file 40 --out output/test_union_${C}.tsv; cp output/test_union_${C}.tsv /content/drive/MyDrive/test_union_${C}.tsv; done
!ls -lh output/test_union_*.tsv

In [ ]:
# 4) Train/val SUBSET ids (stratified 200k/20k) + S1/GT slice files.
import pandas as pd
s1 = pd.read_csv('dataset/train/train_source1.tsv', sep='\t')
gt = pd.read_csv('dataset/train/train_ground_truth.tsv', sep='\t')
gt['k'] = gt['matched_entity_ids'].fillna('').apply(lambda s: 0 if s == '' else s.count(',') + 1)
df = gt.merge(s1[['entity_id', 'country']], left_on='source1_entity_id', right_on='entity_id')
df['stratum'] = df['country'] + '_' + df['k'].clip(upper=6).astype(str)
tr = df.groupby('stratum', group_keys=False).apply(lambda g: g.sample(frac=200000 / len(df), random_state=42))
rest = df.drop(tr.index)
va = rest.groupby('stratum', group_keys=False).apply(lambda g: g.sample(frac=20000 / len(rest), random_state=43))
print('train', len(tr), 'val', len(va))
for name, sub in [('train200k', tr), ('val20k', va)]:
    sub[['source1_entity_id', 'matched_entity_ids']].to_csv(f'sample/{name}_gt.tsv', sep='\t', index=False)
    s1[s1['entity_id'].isin(set(sub['source1_entity_id']))].to_csv(f'sample/{name}_s1.tsv', sep='\t', index=False)
sub = pd.concat([s1[s1['entity_id'].isin(set(tr['source1_entity_id']))], s1[s1['entity_id'].isin(set(va['source1_entity_id']))]])
sub.to_csv('sample/sub220k_s1.tsv', sep='\t', index=False)
print(sub['country'].value_counts().to_dict())

In [ ]:
# 5) Train-subset blocking vs FULL train targets (220k queries x 2 fields), then split by id.
S = "code/business_entity_resolution/src"
!python3 $S/blocking.py --s1 sample/sub220k_s1.tsv --s2 dataset/train/train_source2.tsv --s3 dataset/train/train_source3.tsv --field name --out output/sub220k_name_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US --resume
!python3 $S/blocking.py --s1 sample/sub220k_s1.tsv --s2 dataset/train/train_source2.tsv --s3 dataset/train/train_source3.tsv --field address --out output/sub220k_addr_{country}.tsv --k 40 --min-ngram 3 --max-ngram 4 --min-df 2 --max-df 0.05 --countries India,US --resume
!for C in India US; do python3 $S/union_candidates.py --a output/sub220k_name_${C}.tsv --b output/sub220k_addr_${C}.tsv --k-per-file 40 --out output/sub220k_union_${C}.tsv; done
# split union+channels into train200k / val20k id sets (single combined files)
import pandas as pd
tr_ids = set(pd.read_csv('sample/train200k_gt.tsv', sep='\t')['source1_entity_id'])
va_ids = set(pd.read_csv('sample/val20k_gt.tsv', sep='\t')['source1_entity_id'])
for kind in ['union', 'name', 'addr']:
    df = pd.concat([pd.read_csv(f'output/sub220k_{kind}_{C}.tsv', sep='\t') for C in ['India', 'US']], ignore_index=True)
    df[df['source1_entity_id'].isin(tr_ids)].to_csv(f'output/cands_train200k_{kind}.tsv', sep='\t', index=False)
    df[df['source1_entity_id'].isin(va_ids)].to_csv(f'output/cands_val20k_{kind}.tsv', sep='\t', index=False)
    print(kind, len(df))

In [ ]:
# 6) SCALE-RECALL CHECK: full-target recall on val20k. PASTE OUTPUT. Expect ~0.99.
S = "code/business_entity_resolution/src"
!python3 $S/blocking_recall.py --candidates output/cands_val20k_union.tsv --ground-truth sample/val20k_gt.tsv --s1 sample/val20k_s1.tsv --s2 dataset/train/train_source2.tsv --s3 dataset/train/train_source3.tsv --ks 25,50,80

In [ ]:
# 7) TRAIN on 200k (~1h) -> model + tuned tau. PASTE threshold.json + BEST line.
S = "code/business_entity_resolution/src"
!python3 $S/train.py --s1 sample/sub220k_s1.tsv --s2 dataset/train/train_source2.tsv --s3 dataset/train/train_source3.tsv --train-gt sample/train200k_gt.tsv --val-gt sample/val20k_gt.tsv --candidates output/cands_train200k_union.tsv --cand-name output/cands_train200k_name.tsv --cand-addr output/cands_train200k_addr.tsv --feature-set v2 --out-dir output/full_model --seed 42
!cat output/full_model/threshold.json
!cp output/full_model/model.txt output/full_model/threshold.json /content/drive/MyDrive/

In [ ]:
# 8) TEST INFERENCE per country (~2-3h) -> concat -> VALIDATE. Leave tab open.
S = "code/business_entity_resolution/src"
!for C in India US France; do python3 $S/predict.py --s1 dataset/test/test_source1.tsv --s2 dataset/test/test_source2.tsv --s3 dataset/test/test_source3.tsv --candidates output/test_union_${C}.tsv --cand-name output/test_${C}_name.tsv --cand-addr output/test_${C}_addr.tsv --model output/full_model/model.txt --threshold-json output/full_model/threshold.json --feature-set v2 --country $C --workers 2 --out-matches output/matching_${C}.tsv --out-cands output/candpairs_${C}.tsv; cp output/matching_${C}.tsv output/candpairs_${C}.tsv /content/drive/MyDrive/; done
!head -1 output/matching_India.tsv > output/matching_results.tsv && tail -n +2 -q output/matching_India.tsv output/matching_US.tsv output/matching_France.tsv >> output/matching_results.tsv
!head -1 output/candpairs_India.tsv > output/candidate_pairs.tsv && tail -n +2 -q output/candpairs_India.tsv output/candpairs_US.tsv output/candpairs_France.tsv >> output/candidate_pairs.tsv
!wc -l output/matching_results.tsv output/candidate_pairs.tsv
!python3 utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test
!cp output/matching_results.tsv output/candidate_pairs.tsv /content/drive/MyDrive/

In [ ]:
# 9) Sanity stats + download. PASTE OUTPUT, then upload matching_results.tsv to the Portal.
import pandas as pd
m = pd.read_csv('output/matching_results.tsv', sep='\t')
print(m.shape)
nm = m['matched_entity_ids'].fillna('').apply(lambda s: 0 if s == '' else s.count(',') + 1)
print('mean kept/S1:', round(float(nm.mean()), 3), '| empty %:', round(float((nm == 0).mean() * 100), 2))
print(nm.value_counts().sort_index().head(12).to_dict())
print(m.head(3).to_string())
!ls -lh output/matching_results.tsv
from google.colab import files
files.download('output/matching_results.tsv')

## What to send back

1. Dry-run output (cell 1) → we confirm budget.
2. Scale-recall output (cell 6) → full-target recall.
3. `threshold.json` + BEST line (cell 7) → full-data val score.
4. Cell 9 stats + validator PASS → submission sanity.
5. **Public leaderboard score** after Portal upload.

Then we iterate: France analysis → error slices → targeted Exp6+.